In [1]:

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

teehr.__version__

'0.8.0'

In [2]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=40,
    executor_memory="16g",
    executor_cores=4,
    # The primary/secondary join shuffles several TiB; teehr's default
    # (2x executor cores) leaves ~15GB per task and OOM-kills executors.
    update_configs={"spark.sql.shuffle.partitions": "4096"},
)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:📦 Configuring Spark cluster with container image: None
INFO:teehr.evaluation.spark_session_utils:🔍 Initial spark namespace from ENV: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔍 Connecting to Kubernetes API: https://172.20.0.1:443
INFO:teehr.evaluation.spark_session_utils:🎯 Executor namespace: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔐 Executor service account: spark (in teehr-hub)
INFO:teehr.evaluation.spark_session_utils:🔐 Using in-cluster authentication
INFO:teehr.evaluation.spark_session_utils:🔗 Setting driver host to pod IP: 10.0.1.73
INFO:teehr.evaluation.spark_session_utils:✅ Spark cluster configuration successful!
INFO:teehr.evaluation.spark_session_utils:   - Executor instances: 40
INFO:teehr.evaluation.spark_session_utils:   - Executor memory: 16g
INFO:teehr.evaluation.spark_session_utils:   - Executor cores: 4
INFO:teehr.evaluatio

In [3]:
%%time
ev = teehr.RemoteReadWriteEvaluation(
    spark=spark
)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


CPU times: user 3.81 ms, sys: 169 μs, total: 3.98 ms
Wall time: 711 ms


In [4]:
joined = ev.joined_timeseries_view(
    # Only the USGS observations are evaluated; without this every primary
    # configuration is shuffled through the join.
    primary_filters=[
        {
            "column": "configuration_name",
            "operator": "=",
            "value": "usgs_observations"
        }
    ],
    secondary_filters=[
        {
            "column": "configuration_name",
            "operator": "in",
            "value": ['nwm30_retrospective', "psu_dhbv_hf", "psu_dprms_hf", "psu_dsacsma_hf", "psu_densemble_hf"]
        }
    ]
)

In [5]:
# joined.to_sdf().show()

In [6]:
# years(value_time) rather than months(): this is a retrospective simulation
# record, so monthly partitions land around 10MB each — far below the 512MB
# target file size. secondary_configuration_name keeps the psu_* runs separate from
# nwm30_retrospective.
#
# write_ordered_by declares the table's sort order, but create_or_replace runs
# CREATE TABLE AS SELECT before the ALTER TABLE lands, so the rows written here
# are not yet physically sorted. routine_table_maintenance reads the declared
# sort order and sorts the data on its next run.
try:
    joined.write_to(
        table_name="sim_joined_timeseries",
        write_mode="create_or_replace",
        partition_by=["years(value_time)", "secondary_configuration_name"],
        write_ordered_by=[
            "location_id",
            "secondary_location_id",
            "value_time",
        ],
    )
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    print("Finished attempting to write joined timeseries table.")
    spark.stop()

INFO:teehr.evaluation.dataframe_base:Writing to table: sim_joined_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: sim_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: sim_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.sim_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.sim_joined_timeseries.
INFO:teehr.evaluation.views.joined_timeseries_view:Computing joined timeseries view
INFO:teehr.evaluation.tables.generic_table:Getting table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.primary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.primary_timeseries.
INFO:teehr.evaluation.dataframe_base:Setting filter [{'column': 'configuration_name', 'operator': '=', 'value': 'usgs_observations'}].
INFO:teeh

Finished attempting to write joined timeseries table.


In [7]:
spark.stop()